<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Windows_ThemePack_Studio_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

View this notebook on GitHub: <a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Windows_ThemePack_Studio_Colab.ipynb" target="_parent">Skquark/AEI-Colab-Notebooks/Windows_ThemePack_Studio_Colab.ipynb</a>


# 🪟 Windows 10/11 `.deskthemepack` & Stream Deck / OBS Interactive Soundboard Studio

A dedicated **Desktop Theme Packager, Accent Color Extractor, OBS Browser-Source HTML5 Soundboard Generator & Elgato Stream Deck Asset Bundler** that unifies outputs from across the AEI notebook suite:

- **21-Event Windows Sound Schemes (`.wav` / `.zip`)**: Automatically discovers sound packs generated by [`Stable-Audio-3_Colab.ipynb`](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Stable-Audio-3_Colab.ipynb), [`MOSS-SoundEffect-v2_Colab.ipynb`](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/MOSS-SoundEffect-v2_Colab.ipynb), [`AudioX-Turbo_Colab.ipynb`](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/AudioX-Turbo_Colab.ipynb), [`MiDashengLM-Gen_Colab.ipynb`](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/MiDashengLM-Gen_Colab.ipynb), [`HunyuanVideo-Foley_Colab.ipynb`](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/HunyuanVideo-Foley_Colab.ipynb), [`TangoFlux-EzAudio_Colab.ipynb`](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/TangoFlux-EzAudio_Colab.ipynb), and [`Audio_PostProcessor_Colab.ipynb`](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Audio_PostProcessor_Colab.ipynb).
- **2K / 4K Desktop Wallpapers (`.png` / `.jpg`)**: Automatically discovers widescreen wallpapers generated by [`Qwen-Image-2.1_Colab.ipynb`](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Qwen-Image-2.1_Colab.ipynb) or uploaded by the user.

---

### Architecture & Core Capabilities

```
┌───────────────────────────────────────────────────────────────────────────────────┐
│        Windows 10/11 ThemePack & Soundboard Studio (4 Interactive Tabs)           │
├───────────────────────────────────────────────────────────────────────────────────┤
│ 1. Complete Windows 10/11 .theme & .deskthemepack Builder                         │
│    • Auto-extracts dominant ColorizationColor (0xAARRGGBB) from wallpaper image   │
│    • Maps all 21 official Windows [AppEvents\Schemes\Apps\.Default\*] sound keys  │
│    • Builds genuine .deskthemepack (CAB) + .theme + Install_Complete_Theme.ps1    │
├───────────────────────────────────────────────────────────────────────────────────┤
│ 2. OBS Browser-Source HTML5 Soundboard & Elgato Stream Deck Packager              │
│    • Generates self-contained obs_soundboard.html with embedded WebAudio buffers, │
│      real-time FFT spectrum visualizer, F13–F24 / Numpad hotkeys & loop toggles   │
│    • Renders 72×72 & 144×144 (@2x) neon glass Stream Deck PNG key icons           │
├───────────────────────────────────────────────────────────────────────────────────┤
│ 3. 1080p Animated Desktop Theme & Sound Event Showcase Video Renderer (.mp4)      │
│    • Renders a 1080p MP4 previewing your desktop mockup, toast notifications,     │
│      and real-time waveform as each Windows sound event plays sequentially        │
├───────────────────────────────────────────────────────────────────────────────────┤
│ 4. Batch Drive Sound Pack -> .deskthemepack & Soundboard Converter                │
└───────────────────────────────────────────────────────────────────────────────────┘
```

| Feature | Specification | Compatibility |
| :--- | :--- | :--- |
| **Theme Format** | `.deskthemepack` (Microsoft Cabinet `MSCF`), `.theme` INI, `.reg` & `Install_Complete_Theme.ps1` | Windows 10 & Windows 11 (22H2 / 23H2 / 24H2) |
| **Audio Format** | `44.1 kHz` / `48 kHz` `16-bit PCM` stereo `.wav` with fade-in/out de-click & `-16 LUFS` normalization | Native Windows `PlaySound` API + WebAudio |
| **Soundboard** | Zero-dependency single-file `obs_soundboard.html` + `72×72` / `144×144` Stream Deck icons | OBS Studio Browser Source, Desktop Chrome/Edge, Touchscreens |


In [ ]:
# @title STEP 1: Mount Google Drive & Install ThemePack / Audio / Cabinet Utilities { display-mode: "form" }
# @markdown Mounts Google Drive (`HF_HOME` at `/content/drive/MyDrive/HF_Cache`), installs `lcab` / `gcab` + `ffmpeg` for native `.deskthemepack` cabinet creation and video rendering, and installs `soundfile`, `scipy`, `pyloudnorm`, `pillow`, and `gradio==5.49.1`.

import os
import sys
import subprocess
import shutil
from pathlib import Path
from tqdm.auto import tqdm
from IPython.display import clear_output

MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
        print("✅ Google Drive mounted at /content/drive")
    except Exception as e:
        print(f"⚠️ Google Drive mount skipped or unavailable ({e}). Using local /content storage.")

DRIVE_ROOT = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content/drive_fallback"
HF_CACHE_DIR = os.path.join(DRIVE_ROOT, "HF_Cache")
SHARED_TTS_ROOT = os.path.join(DRIVE_ROOT, "AEI_TTS_Out")
SHARED_3D_ROOT = os.path.join(DRIVE_ROOT, "AEI_3D_Out")
OUTPUT_DIR = os.path.join(SHARED_TTS_ROOT, "Windows_ThemePacks")

for d in [HF_CACHE_DIR, SHARED_TTS_ROOT, SHARED_3D_ROOT, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")

# Install system cabinet builders (lcab / gcab) and ffmpeg if on Linux/Colab
if sys.platform.startswith("linux"):
    try:
        subprocess.run(["apt-get", "update", "-qq"], check=False, capture_output=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "lcab", "gcab", "ffmpeg"], check=False, capture_output=True)
    except Exception as e:
        print(f"ℹ️ System package check note: {e}")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "soundfile", "scipy", "numpy", "pillow", "pyloudnorm",
        "tqdm", "huggingface_hub", "gradio==5.49.1",
    ],
    check=False,
)

print(f"✅ Environment ready! Output directory: {OUTPUT_DIR}")


In [ ]:
# @title STEP 2: Discover Existing Sound Packs, WAV Stems & Wallpapers Across Google Drive { display-mode: "form" }
# @markdown Scans `/content/drive/MyDrive/AEI_TTS_Out` and `/content/drive/MyDrive/AEI_3D_Out` for generated Windows Sound Scheme `.zip` archives, individual `.wav` files, and 2K/4K wallpaper images.

import os
import zipfile
from pathlib import Path

DEFAULT_ACOUSTIC_PALETTE = "Fluent Glass & Crystal Chimes (Windows 11 Modern)"  # @param ["Fluent Glass & Crystal Chimes (Windows 11 Modern)", "Cyberpunk Neon FM Synth & Sub Glitch", "Warm Acoustic Marimba & Felt Piano", "Sci-Fi Starship Bridge Telemetry", "Minimalist Tactile Micro-Clicks"]
DEFAULT_WALLPAPER_STYLE = "Fluent Sapphire & Violet Acrylic Waves (2560×1440)"  # @param ["Fluent Sapphire & Violet Acrylic Waves (2560×1440)", "Cyberpunk Midnight Neon Horizon (2560×1440)", "Nordic Aurora Borealis Dark Slate (2560×1440)", "Solar Gold & Obsidian Geometric Glass (2560×1440)"]

def discover_drive_sound_packs() -> list:
    '''Finds all .zip sound packs and folders containing .wav files in AEI_TTS_Out and AEI_3D_Out.'''
    roots = [
        globals().get("SHARED_TTS_ROOT", "/content/drive/MyDrive/AEI_TTS_Out"),
        globals().get("SHARED_3D_ROOT", "/content/drive/MyDrive/AEI_3D_Out"),
    ]
    zips = []
    seen = set()
    for r in roots:
        if not os.path.exists(r):
            continue
        for root_dir, _, files in os.walk(r):
            if "HF_Cache" in root_dir:
                continue
            for f in files:
                if f.lower().endswith(".zip"):
                    p = os.path.abspath(os.path.join(root_dir, f))
                    if p not in seen:
                        seen.add(p)
                        zips.append(p)
    zips.sort(key=lambda p: os.path.getmtime(p) if os.path.exists(p) else 0, reverse=True)
    return zips[:150]


def discover_drive_wav_files() -> list:
    '''Finds individual .wav / .mp3 / .flac audio files across AEI_TTS_Out and AEI_3D_Out.'''
    roots = [
        globals().get("SHARED_TTS_ROOT", "/content/drive/MyDrive/AEI_TTS_Out"),
        globals().get("SHARED_3D_ROOT", "/content/drive/MyDrive/AEI_3D_Out"),
    ]
    wavs = []
    seen = set()
    for r in roots:
        if not os.path.exists(r):
            continue
        for root_dir, _, files in os.walk(r):
            if "HF_Cache" in root_dir:
                continue
            for f in files:
                if os.path.splitext(f)[1].lower() in {".wav", ".mp3", ".flac", ".ogg"}:
                    p = os.path.abspath(os.path.join(root_dir, f))
                    if p not in seen:
                        seen.add(p)
                        wavs.append(p)
    wavs.sort(key=lambda p: os.path.getmtime(p) if os.path.exists(p) else 0, reverse=True)
    return wavs[:300]


def discover_drive_wallpapers() -> list:
    '''Finds generated wallpapers and images across AEI_3D_Out and AEI_TTS_Out.'''
    roots = [
        globals().get("SHARED_3D_ROOT", "/content/drive/MyDrive/AEI_3D_Out"),
        globals().get("SHARED_TTS_ROOT", "/content/drive/MyDrive/AEI_TTS_Out"),
    ]
    imgs = []
    seen = set()
    for r in roots:
        if not os.path.exists(r):
            continue
        for root_dir, _, files in os.walk(r):
            if "HF_Cache" in root_dir or "_thumbs" in root_dir:
                continue
            for f in files:
                if os.path.splitext(f)[1].lower() in {".png", ".jpg", ".jpeg", ".webp"}:
                    p = os.path.abspath(os.path.join(root_dir, f))
                    if p not in seen:
                        seen.add(p)
                        imgs.append(p)
    imgs.sort(key=lambda p: os.path.getmtime(p) if os.path.exists(p) else 0, reverse=True)
    return imgs[:200]


found_packs = discover_drive_sound_packs()
found_wavs = discover_drive_wav_files()
found_walls = discover_drive_wallpapers()
print(f"📦 Discovered {len(found_packs)} Sound Pack .zip archives in Google Drive")
print(f"🔊 Discovered {len(found_wavs)} individual Audio/SFX files in Google Drive")
print(f"🖼️ Discovered {len(found_walls)} Wallpaper/Image files in Google Drive")


In [ ]:
# @title STEP 3: Core Engines (.theme / .deskthemepack Builder, Accent Color Extractor, HTML5 Soundboard & Showcase Video) { display-mode: "form" }
# @markdown Implements the 21-event Windows Sound Scheme registry mapper, dominant `ColorizationColor` extractor, `.deskthemepack` cabinet bundler, self-contained WebAudio `obs_soundboard.html` + Stream Deck icon renderer, and 1080p video showcase generator.

import os
import io
import re
import time
import math
import json
import base64
import struct
import shutil
import zipfile
import tempfile
import subprocess
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import soundfile as sf
from scipy import signal
from PIL import Image, ImageDraw, ImageFont, ImageFilter

# ---------------------------------------------------------------------------
# 1. Official 21 Windows 10/11 Sound Scheme Events & Registry Paths
# ---------------------------------------------------------------------------
WINDOWS_21_EVENTS: List[Dict[str, Any]] = [
    {"id": "WindowsLogon", "label": "Windows Logon / Startup", "reg_app": ".Default", "freq_hz": 523.25, "dur_s": 1.65, "chord": [1.0, 1.25, 1.5, 2.0]},
    {"id": "WindowsLogoff", "label": "Windows Logoff / Shutdown", "reg_app": ".Default", "freq_hz": 440.0, "dur_s": 1.35, "chord": [2.0, 1.5, 1.25, 1.0]},
    {"id": "WindowsUnlock", "label": "Windows Unlock", "reg_app": ".Default", "freq_hz": 587.33, "dur_s": 0.75, "chord": [1.0, 1.5, 2.0]},
    {"id": "SystemNotification", "label": "System Notification Toast", "reg_app": ".Default", "freq_hz": 659.25, "dur_s": 0.68, "chord": [1.0, 1.25, 1.5]},
    {"id": "SystemAsterisk", "label": "Asterisk / Info Alert", "reg_app": ".Default", "freq_hz": 783.99, "dur_s": 0.55, "chord": [1.0, 1.33]},
    {"id": "SystemExclamation", "label": "Exclamation / Warning", "reg_app": ".Default", "freq_hz": 493.88, "dur_s": 0.65, "chord": [1.0, 1.189, 1.5]},
    {"id": "SystemHand", "label": "Critical Stop / Error", "reg_app": ".Default", "freq_hz": 220.0, "dur_s": 0.70, "chord": [1.0, 1.059]},
    {"id": "SystemQuestion", "label": "System Question Prompt", "reg_app": ".Default", "freq_hz": 587.33, "dur_s": 0.55, "chord": [1.0, 1.33, 1.68]},
    {"id": ".Default", "label": "Default Beep / Click", "reg_app": ".Default", "freq_hz": 698.46, "dur_s": 0.35, "chord": [1.0, 1.5]},
    {"id": "DeviceConnect", "label": "USB / Hardware Device Connect", "reg_app": ".Default", "freq_hz": 523.25, "dur_s": 0.58, "chord": [1.0, 1.33, 1.68]},
    {"id": "DeviceDisconnect", "label": "USB / Hardware Device Disconnect", "reg_app": ".Default", "freq_hz": 523.25, "dur_s": 0.58, "chord": [1.5, 1.189, 1.0]},
    {"id": "DeviceFail", "label": "Hardware Device Fail", "reg_app": ".Default", "freq_hz": 293.66, "dur_s": 0.62, "chord": [1.0, 0.94, 0.89]},
    {"id": "LowBatteryAlarm", "label": "Low Battery Alarm", "reg_app": ".Default", "freq_hz": 440.0, "dur_s": 0.65, "chord": [1.0, 0.89, 1.0]},
    {"id": "CriticalBatteryAlarm", "label": "Critical Battery Alarm", "reg_app": ".Default", "freq_hz": 329.63, "dur_s": 0.75, "chord": [1.0, 0.84, 0.84]},
    {"id": "MailBeep", "label": "New Mail Arrival Chime", "reg_app": ".Default", "freq_hz": 880.0, "dur_s": 0.60, "chord": [1.0, 1.25]},
    {"id": "MessageNudge", "label": "Instant Message Nudge", "reg_app": ".Default", "freq_hz": 740.0, "dur_s": 0.45, "chord": [1.0, 1.5, 1.0]},
    {"id": "Notification.Default", "label": "Action Center Notification", "reg_app": ".Default", "freq_hz": 659.25, "dur_s": 0.62, "chord": [1.0, 1.5]},
    {"id": "Notification.IM", "label": "Chat / IM Notification", "reg_app": ".Default", "freq_hz": 783.99, "dur_s": 0.48, "chord": [1.0, 1.25]},
    {"id": "Notification.Mail", "label": "Mail Toast Notification", "reg_app": ".Default", "freq_hz": 698.46, "dur_s": 0.68, "chord": [1.0, 1.25, 1.5]},
    {"id": "Notification.Reminder", "label": "Calendar Reminder Chime", "reg_app": ".Default", "freq_hz": 587.33, "dur_s": 0.85, "chord": [1.0, 1.25, 1.5, 1.875]},
    {"id": "EmptyRecycleBin", "label": "Empty Recycle Bin Crisp Whoosh", "reg_app": "Explorer", "freq_hz": 392.0, "dur_s": 0.52, "chord": [1.5, 1.2, 0.8]},
]


# ---------------------------------------------------------------------------
# 2. Procedural 44.1 kHz Stereo Sound Scheme & 2K Wallpaper Synthesizers
# ---------------------------------------------------------------------------
def synthesize_event_wav(
    event_spec: Dict[str, Any],
    palette_name: str = "Fluent Glass & Crystal Chimes (Windows 11 Modern)",
    sr: int = 44100,
) -> np.ndarray:
    '''Synthesizes a polished 44.1 kHz stereo 16-bit-ready waveform for a Windows sound event.'''
    dur = float(event_spec.get("dur_s", 0.65))
    base_f = float(event_spec.get("freq_hz", 523.25))
    chord = event_spec.get("chord", [1.0, 1.5])
    n_samples = int(sr * dur)
    t = np.linspace(0.0, dur, n_samples, endpoint=False, dtype=np.float32)

    left = np.zeros(n_samples, dtype=np.float32)
    right = np.zeros(n_samples, dtype=np.float32)

    note_spacing = min(0.11, dur / max(2, len(chord) + 1))
    for i, ratio in enumerate(chord):
        onset = int(i * note_spacing * sr)
        rem = n_samples - onset
        if rem <= 0:
            continue
        ti = np.linspace(0.0, rem / sr, rem, endpoint=False, dtype=np.float32)
        f = base_f * float(ratio)

        if "Cyberpunk" in palette_name:
            mod = np.sin(2.0 * np.pi * (f * 2.0) * ti) * np.exp(-ti * 9.0) * 1.8
            sig = np.sin(2.0 * np.pi * f * ti + mod) + 0.35 * np.sin(2.0 * np.pi * (f * 0.5) * ti)
            env = (1.0 - np.exp(-ti * 180.0)) * np.exp(-ti * 6.5)
        elif "Marimba" in palette_name:
            sig = np.sin(2.0 * np.pi * f * ti) + 0.25 * np.sin(2.0 * np.pi * (f * 4.0) * ti) * np.exp(-ti * 28.0)
            env = (1.0 - np.exp(-ti * 260.0)) * np.exp(-ti * 8.5)
        elif "Starship" in palette_name:
            f_glide = f * (1.0 + 0.06 * np.exp(-ti * 18.0))
            sig = np.sin(2.0 * np.pi * f_glide * ti) + 0.4 * np.sin(2.0 * np.pi * (f * 1.5) * ti)
            env = (1.0 - np.exp(-ti * 220.0)) * np.exp(-ti * 7.0)
        elif "Minimalist" in palette_name:
            sig = np.sin(2.0 * np.pi * f * ti) * np.exp(-ti * 22.0)
            env = (1.0 - np.exp(-ti * 400.0)) * np.exp(-ti * 16.0)
        else:
            # Fluent Glass & Crystal Chimes
            sig = (
                np.sin(2.0 * np.pi * f * ti)
                + 0.42 * np.sin(2.0 * np.pi * (f * 2.0) * ti + 0.3) * np.exp(-ti * 5.0)
                + 0.18 * np.sin(2.0 * np.pi * (f * 3.01) * ti) * np.exp(-ti * 11.0)
            )
            env = (1.0 - np.exp(-ti * 140.0)) * np.exp(-ti * 4.8)

        pan = 0.5 + 0.22 * math.sin((i + 1) * 1.3)
        left[onset:] += sig * env * math.sqrt(1.0 - pan)
        right[onset:] += sig * env * math.sqrt(pan)

    stereo = np.stack([left, right], axis=-1)
    # 5ms fade-in and 15ms fade-out to guarantee zero clicks in Windows PlaySound
    fade_in = min(int(0.005 * sr), n_samples // 4)
    fade_out = min(int(0.015 * sr), n_samples // 3)
    if fade_in > 0:
        stereo[:fade_in] *= np.linspace(0.0, 1.0, fade_in, dtype=np.float32)[:, None]
    if fade_out > 0:
        stereo[-fade_out:] *= np.linspace(1.0, 0.0, fade_out, dtype=np.float32)[:, None]

    peak = float(np.max(np.abs(stereo))) if stereo.size else 1.0
    if peak > 1e-6:
        stereo = stereo / peak * 0.82
    return stereo.astype(np.float32)


def generate_procedural_wallpaper(
    style_name: str = "Fluent Sapphire & Violet Acrylic Waves (2560×1440)",
    width: int = 2560,
    height: int = 1440,
) -> Image.Image:
    '''Generates a smooth 2560×1440 desktop wallpaper with layered ribbons and acrylic glow.'''
    y = np.linspace(-1.0, 1.0, height, dtype=np.float32)[:, None]
    x = np.linspace(-1.0, 1.0, width, dtype=np.float32)[None, :]

    if "Cyberpunk" in style_name:
        c_bg1 = np.array([12, 10, 28], dtype=np.float32)
        c_bg2 = np.array([34, 14, 58], dtype=np.float32)
        c_wave1 = np.array([0, 240, 255], dtype=np.float32)
        c_wave2 = np.array([255, 42, 145], dtype=np.float32)
    elif "Aurora" in style_name:
        c_bg1 = np.array([9, 18, 26], dtype=np.float32)
        c_bg2 = np.array([18, 38, 48], dtype=np.float32)
        c_wave1 = np.array([45, 235, 165], dtype=np.float32)
        c_wave2 = np.array([70, 155, 255], dtype=np.float32)
    elif "Solar" in style_name:
        c_bg1 = np.array([16, 15, 18], dtype=np.float32)
        c_bg2 = np.array([38, 28, 22], dtype=np.float32)
        c_wave1 = np.array([255, 175, 55], dtype=np.float32)
        c_wave2 = np.array([235, 85, 45], dtype=np.float32)
    else:
        # Fluent Sapphire & Violet
        c_bg1 = np.array([10, 18, 38], dtype=np.float32)
        c_bg2 = np.array([24, 34, 68], dtype=np.float32)
        c_wave1 = np.array([56, 152, 255], dtype=np.float32)
        c_wave2 = np.array([155, 88, 255], dtype=np.float32)

    t_grad = np.clip((y + 1.0) * 0.5, 0.0, 1.0)[..., None]
    canvas = (1.0 - t_grad) * c_bg1 + t_grad * c_bg2

    # Layered cosine acrylic ribbons
    wave_a = np.exp(-((y - 0.18 * np.sin(x * 2.6 + 0.4) - 0.12 * np.cos(x * 1.4)) ** 2) / 0.045)[..., None]
    wave_b = np.exp(-((y + 0.22 * np.cos(x * 2.1 - 0.6) - 0.05) ** 2) / 0.065)[..., None]

    canvas = canvas * (1.0 - 0.65 * wave_a) + c_wave1 * (0.65 * wave_a)
    canvas = canvas * (1.0 - 0.55 * wave_b) + c_wave2 * (0.55 * wave_b)

    rgb = np.clip(canvas, 0, 255).astype(np.uint8)
    return Image.fromarray(rgb, mode="RGB")


# ---------------------------------------------------------------------------
# 3. Dominant Wallpaper Accent Color Extractor & Windows 11 Desktop Preview
# ---------------------------------------------------------------------------
def extract_windows_accent_color(pil_img: Image.Image) -> Tuple[str, str, List[str]]:
    '''
    Extracts a vibrant dominant accent color from a wallpaper image and returns:
      - hex_color: '#RRGGBB'
      - win_dword_str: '0xC4RRGGBB' (for Windows [VisualStyles] ColorizationColor)
      - palette_hex_list: top 5 palette hex colors
    '''
    small = pil_img.convert("RGB").resize((96, 96), Image.Resampling.BILINEAR)
    quant = small.quantize(colors=6, method=Image.Quantize.MEDIANCUT).convert("RGB")
    arr = np.asarray(quant, dtype=np.float32).reshape(-1, 3)

    unique_colors, counts = np.unique(arr, axis=0, return_counts=True)
    order = np.argsort(-counts)
    unique_colors = unique_colors[order]
    counts = counts[order]

    best_rgb = (56, 152, 255)
    best_score = -1.0
    palette_hex = []

    for c, cnt in zip(unique_colors, counts):
        r, g, b = int(c[0]), int(c[1]), int(c[2])
        palette_hex.append(f"#{r:02X}{g:02X}{b:02X}")
        mx = max(r, g, b) / 255.0
        mn = min(r, g, b) / 255.0
        sat = (mx - mn) / max(1e-5, mx)
        val = mx
        # Prefer saturated, mid-to-bright colors suitable for Windows UI accents
        score = (sat + 0.15) * (1.0 - abs(val - 0.72)) * math.sqrt(float(cnt))
        if score > best_score:
            best_score = score
            best_rgb = (r, g, b)

    r, g, b = best_rgb
    hex_str = f"#{r:02X}{g:02X}{b:02X}"
    win_dword = f"0xC4{r:02X}{g:02X}{b:02X}"
    return hex_str, win_dword, palette_hex[:5]


def render_windows11_desktop_mockup(
    wallpaper_img: Image.Image,
    theme_display_name: str,
    accent_hex: str,
    palette_hexes: List[str],
    dark_mode: bool = True,
    active_event_label: str = "SystemNotification — Windows 11 Theme Active",
) -> Image.Image:
    '''Renders a 1280×720 Windows 11 Desktop, Centered Taskbar & Toast Notification mockup.'''
    w, h = 1280, 720
    canvas = wallpaper_img.convert("RGB").resize((w, h), Image.Resampling.LANCZOS).convert("RGBA")
    draw = ImageDraw.Draw(canvas, "RGBA")

    hex_clean = accent_hex.lstrip("#")
    ar, ag, ab = (
        int(hex_clean[0:2], 16),
        int(hex_clean[2:4], 16),
        int(hex_clean[4:6], 16),
    ) if len(hex_clean) == 6 else (56, 152, 255)

    panel_bg = (20, 22, 30, 215) if dark_mode else (242, 245, 250, 225)
    text_fg = (240, 244, 255, 255) if dark_mode else (24, 28, 36, 255)
    sub_fg = (175, 185, 205, 255) if dark_mode else (80, 90, 110, 255)

    # Bottom Windows 11 Taskbar
    tb_h = 48
    draw.rectangle([0, h - tb_h, w, h], fill=panel_bg)
    draw.line([(0, h - tb_h), (w, h - tb_h)], fill=(ar, ag, ab, 180), width=2)

    # Centered taskbar icons
    cx = w // 2
    for idx in range(-3, 4):
        ix = cx + idx * 42
        iy = h - tb_h // 2
        col = (ar, ag, ab, 255) if idx == -3 else (sub_fg[0], sub_fg[1], sub_fg[2], 190)
        draw.rounded_rectangle([ix - 13, iy - 13, ix + 13, iy + 13], radius=6, fill=col)

    # Bottom-Right Windows 11 Toast Notification Card
    tw, th = 390, 118
    tx0, ty0 = w - tw - 24, h - tb_h - th - 20
    draw.rounded_rectangle([tx0, ty0, tx0 + tw, ty0 + th], radius=12, fill=panel_bg, outline=(ar, ag, ab, 230), width=2)
    draw.rounded_rectangle([tx0 + 16, ty0 + 18, tx0 + 24, ty0 + th - 18], radius=4, fill=(ar, ag, ab, 255))
    draw.text((tx0 + 38, ty0 + 20), f"AEI ThemePack: {theme_display_name}", fill=text_fg)
    draw.text((tx0 + 38, ty0 + 48), f"🔊 {active_event_label}", fill=(ar, ag, ab, 255))
    draw.text((tx0 + 38, ty0 + 76), f"Accent Color: {accent_hex}  |  21 WAV Events Ready", fill=sub_fg)

    # Top-Left Palette Swatch Pill
    draw.rounded_rectangle([20, 20, 350, 74], radius=10, fill=panel_bg, outline=(ar, ag, ab, 180), width=1)
    draw.text((34, 38), "Extracted Palette:", fill=text_fg)
    for i, p_hex in enumerate(palette_hexes[:5]):
        pc = p_hex.lstrip("#")
        pr, pg, pb = int(pc[0:2], 16), int(pc[2:4], 16), int(pc[4:6], 16)
        sx = 175 + i * 32
        draw.rounded_rectangle([sx, 31, sx + 24, 55], radius=5, fill=(pr, pg, pb, 255), outline=(255, 255, 255, 160), width=1)

    return canvas.convert("RGB")


# ---------------------------------------------------------------------------
# 4. Core Engine 1: Complete Windows 10/11 .theme & .deskthemepack Bundler
# ---------------------------------------------------------------------------
def build_windows_themepack_bundle(
    theme_name: str = "AEI Fluent Cyber Glass",
    drive_sound_zip: str = "(None — Synthesize Procedural 21-Event Scheme)",
    uploaded_sound_files: Optional[List[Any]] = None,
    acoustic_palette: str = "Fluent Glass & Crystal Chimes (Windows 11 Modern)",
    wallpaper_upload: Optional[Image.Image] = None,
    drive_wallpaper_path: str = "(None — Generate Procedural 2K Wallpaper)",
    procedural_wallpaper_style: str = "Fluent Sapphire & Violet Acrylic Waves (2560×1440)",
    extra_slideshow_wallpapers: Optional[List[Any]] = None,
    override_accent_hex: str = "",
    system_dark_mode: bool = True,
    slideshow_interval_min: int = 30,
) -> Tuple[Image.Image, str, str, str, str]:
    '''
    Builds a complete Windows 10/11 Desktop Theme & 21-Event Sound Scheme package:
      1. Extracts or synthesizes all 21 Windows sound events in 44.1 kHz 16-bit PCM stereo .wav
      2. Prepares DesktopBackground/ wallpapers and extracts ColorizationColor (0xAARRGGBB)
      3. Writes compliant <ThemeSlug>.theme INI + Install_Complete_Theme.ps1 + Install_SoundScheme.reg
      4. Packages both a genuine .deskthemepack (via lcab/gcab or ZIP/CAB) and a universal .zip bundle
    Returns:
      (desktop_mockup_pil, preview_logon_wav_path, deskthemepack_path, zip_bundle_path, status_markdown)
    '''
    t0 = time.time()
    clean_title = (theme_name or "AEI Custom Theme").strip()
    slug = re.sub(r"[^A-Za-z0-9_-]+", "_", clean_title).strip("_") or "AEI_Theme"
    scheme_key = slug[:28]

    out_root = globals().get("OUTPUT_DIR", "/content/drive/MyDrive/AEI_TTS_Out/Windows_ThemePacks")
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    build_dir = os.path.join(out_root, f"{slug}_{ts}")
    bg_dir = os.path.join(build_dir, "DesktopBackground")
    media_dir = os.path.join(build_dir, "Media")
    os.makedirs(bg_dir, exist_ok=True)
    os.makedirs(media_dir, exist_ok=True)

    # --- Step A: Prepare Wallpaper(s) ---
    wallpapers_saved: List[str] = []
    if wallpaper_upload is not None:
        primary_wall = wallpaper_upload.convert("RGB")
    elif drive_wallpaper_path and os.path.exists(drive_wallpaper_path):
        primary_wall = Image.open(drive_wallpaper_path).convert("RGB")
    else:
        primary_wall = generate_procedural_wallpaper(style_name=procedural_wallpaper_style)

    w1_name = f"{slug}_wallpaper_01.png"
    w1_path = os.path.join(bg_dir, w1_name)
    primary_wall.save(w1_path, format="PNG")
    wallpapers_saved.append(w1_name)

    if extra_slideshow_wallpapers:
        for idx, wf in enumerate(extra_slideshow_wallpapers[:10], start=2):
            wp = wf.name if hasattr(wf, "name") else str(wf)
            if os.path.exists(wp):
                try:
                    im = Image.open(wp).convert("RGB")
                    wn = f"{slug}_wallpaper_{idx:02d}.png"
                    im.save(os.path.join(bg_dir, wn), format="PNG")
                    wallpapers_saved.append(wn)
                except Exception:
                    pass

    auto_hex, auto_dword, palette_hexes = extract_windows_accent_color(primary_wall)
    if override_accent_hex and len(override_accent_hex.strip().lstrip("#")) == 6:
        hc = override_accent_hex.strip().lstrip("#").upper()
        accent_hex = f"#{hc}"
        accent_dword = f"0xC4{hc}"
    else:
        accent_hex = auto_hex
        accent_dword = auto_dword

    # --- Step B: Populate all 21 Windows Sound Scheme .wav files ---
    source_wav_Pool: Dict[str, str] = {}
    temp_extract_dir = tempfile.mkdtemp(prefix="aei_sfx_extract_")

    if drive_sound_zip and os.path.exists(drive_sound_zip) and zipfile.is_zipfile(drive_sound_zip):
        with zipfile.ZipFile(drive_sound_zip, "r") as zf:
            zf.extractall(temp_extract_dir)
        for r_dir, _, files in os.walk(temp_extract_dir):
            for f in files:
                if f.lower().endswith(".wav"):
                    source_wav_Pool[os.path.splitext(f)[0].lower()] = os.path.join(r_dir, f)

    if uploaded_sound_files:
        for uf in uploaded_sound_files:
            up_p = uf.name if hasattr(uf, "name") else str(uf)
            if os.path.exists(up_p):
                if up_p.lower().endswith(".zip") and zipfile.is_zipfile(up_p):
                    with zipfile.ZipFile(up_p, "r") as zf:
                        zf.extractall(temp_extract_dir)
                    for r_dir, _, files in os.walk(temp_extract_dir):
                        for f in files:
                            if f.lower().endswith(".wav"):
                                source_wav_Pool[os.path.splitext(f)[0].lower()] = os.path.join(r_dir, f)
                elif up_p.lower().endswith(".wav"):
                    source_wav_Pool[os.path.splitext(os.path.basename(up_p))[0].lower()] = up_p

    event_wav_map: Dict[str, str] = {}
    matched_from_pack = 0

    for ev in WINDOWS_21_EVENTS:
        ev_id = ev["id"]
        safe_ev = ev_id.lstrip(".")
        target_wav_name = f"{slug}_{safe_ev}.wav"
        target_wav_path = os.path.join(media_dir, target_wav_name)

        # Match by event ID substring in source_wav_Pool
        matched_src = None
        for k_lower, src_p in source_wav_Pool.items():
            if safe_ev.lower() in k_lower or ev_id.lower() == k_lower:
                matched_src = src_p
                break

        if matched_src and os.path.exists(matched_src):
            try:
                data, sr_in = sf.read(matched_src, always_2d=True)
                sf.write(target_wav_path, data, sr_in, subtype="PCM_16")
                matched_from_pack += 1
            except Exception:
                wav_arr = synthesize_event_wav(ev, palette_name=acoustic_palette, sr=44100)
                sf.write(target_wav_path, wav_arr, 44100, subtype="PCM_16")
        else:
            wav_arr = synthesize_event_wav(ev, palette_name=acoustic_palette, sr=44100)
            sf.write(target_wav_path, wav_arr, 44100, subtype="PCM_16")

        event_wav_map[ev_id] = target_wav_name

    shutil.rmtree(temp_extract_dir, ignore_errors=True)

    # --- Step C: Generate Compliant Windows 10/11 .theme File ---
    mode_str = "Dark" if system_dark_mode else "Light"
    slideshow_lines = [
        "[Slideshow]",
        f"Interval={int(slideshow_interval_min) * 60000}",
        "Shuffle=0",
        f"ImagesRootPath=DesktopBackground",
    ] + [f"Item{idx}Path=DesktopBackground\\{wn}" for idx, wn in enumerate(wallpapers_saved)]

    sound_ini_lines = [
        "[Sounds]",
        f"; Computer-generated Windows 10/11 Sound Scheme: {clean_title}",
        f"SchemeName={clean_title}",
        "",
    ]
    for ev in WINDOWS_21_EVENTS:
        ev_id = ev["id"]
        reg_app = ev["reg_app"]
        wav_fn = event_wav_map[ev_id]
        sound_ini_lines.extend([
            f"[AppEvents\\Schemes\\Apps\\{reg_app}\\{ev_id}\\.Current]",
            f"DefaultValue=Media\\{wav_fn}",
            "",
        ])

    theme_ini_content = "\r\n".join([
        "; Copyright (c) AEI Colab Studio — Windows 10/11 Desktop Theme",
        "[Theme]",
        f"DisplayName={clean_title}",
        "ThemeId={88E41B2A-4D79-4E31-9B12-AE1C01AB2026}",
        "",
        "[Control Panel\\Desktop]",
        f"Wallpaper=DesktopBackground\\{wallpapers_saved[0]}",
        "TileWallpaper=0",
        "WallpaperStyle=10",
        "Pattern=",
        "",
        "[VisualStyles]",
        "Path=%SystemRoot%\\resources\\themes\\Aero\\Aero.msstyles",
        "ColorStyle=NormalColor",
        "Size=NormalSize",
        "AutoColorization=0",
        f"ColorizationColor={accent_dword}",
        f"SystemMode={mode_str}",
        f"AppMode={mode_str}",
        "",
        *slideshow_lines,
        "",
        *sound_ini_lines,
        "[MasterThemeSelector]",
        "MTSM=RJSPBS",
        "",
    ])

    theme_file_path = os.path.join(build_dir, f"{slug}.theme")
    with open(theme_file_path, "w", encoding="utf-8", newline="") as f:
        f.write(theme_ini_content)

    # --- Step D: Generate PowerShell 1-Click Installer & .reg File ---
    reg_lines = [
        "Windows Registry Editor Version 5.00",
        "",
        f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Names\\{scheme_key}]",
        f'@="{clean_title}"',
        "",
    ]
    for ev in WINDOWS_21_EVENTS:
        ev_id = ev["id"]
        reg_app = ev["reg_app"]
        wav_fn = event_wav_map[ev_id]
        win_target_path = f"C:\\\\Windows\\\\Media\\\\{slug}\\\\{wav_fn}"
        reg_lines.extend([
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{reg_app}\\{ev_id}\\{scheme_key}]",
            f'@="{win_target_path}"',
            "",
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{reg_app}\\{ev_id}\\.Current]",
            f'@="{win_target_path}"',
            "",
        ])

    reg_path = os.path.join(build_dir, f"Install_{slug}_Sounds.reg")
    with open(reg_path, "w", encoding="utf-8", newline="\r\n") as f:
        f.write("\r\n".join(reg_lines))

    ps1_content = "\r\n".join([
        f"# 1-Click Windows 10/11 Theme & Sound Scheme Installer — {clean_title}",
        "$ErrorActionPreference = 'Stop'",
        "$ScriptDir = Split-Path -Parent $MyInvocation.MyCommand.Path",
        f"$ThemeDest = Join-Path $env:LOCALAPPDATA 'Microsoft\\Windows\\Themes\\{slug}'",
        "New-Item -ItemType Directory -Force -Path (Join-Path $ThemeDest 'DesktopBackground') | Out-Null",
        "New-Item -ItemType Directory -Force -Path (Join-Path $ThemeDest 'Media') | Out-Null",
        "Copy-Item -Path (Join-Path $ScriptDir 'DesktopBackground\\*') -Destination (Join-Path $ThemeDest 'DesktopBackground') -Force",
        "Copy-Item -Path (Join-Path $ScriptDir 'Media\\*') -Destination (Join-Path $ThemeDest 'Media') -Force",
        f"$ThemeFile = Join-Path $ThemeDest '{slug}.theme'",
        f"(Get-Content (Join-Path $ScriptDir '{slug}.theme')) "
        f"-replace 'DesktopBackground\\\\', \"$ThemeDest\\DesktopBackground\\\" "
        f"-replace 'Media\\\\', \"$ThemeDest\\Media\\\" | Set-Content -Encoding UTF8 $ThemeFile",
        "Write-Host 'Applying Windows Theme & 21-Event Sound Scheme...' -ForegroundColor Cyan",
        "Start-Process -FilePath $ThemeFile",
        "Write-Host 'Done! Your wallpaper, accent color, and 21 sound events are now active.' -ForegroundColor Green",
    ])
    ps1_path = os.path.join(build_dir, "Install_Complete_Theme.ps1")
    with open(ps1_path, "w", encoding="utf-8", newline="") as f:
        f.write(ps1_content)

    # --- Step E: Package .deskthemepack (CAB / Archive) & .zip Bundle ---
    deskthemepack_path = os.path.join(out_root, f"{slug}_{ts}.deskthemepack")
    zip_bundle_path = os.path.join(out_root, f"{slug}_{ts}_Complete_ThemePack.zip")

    with zipfile.ZipFile(zip_bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for r_dir, _, files in os.walk(build_dir):
            for fn in files:
                full_p = os.path.join(r_dir, fn)
                rel_p = os.path.relpath(full_p, build_dir)
                zf.write(full_p, arcname=rel_p)

    # Try gcab or lcab to create a genuine Microsoft Cabinet (.deskthemepack); fallback to cabinet-compatible archive copy
    cab_ok = False
    if shutil.which("gcab"):
        res = subprocess.run(
            ["gcab", "-c", deskthemepack_path, f"{slug}.theme", "DesktopBackground", "Media"],
            cwd=build_dir,
            capture_output=True,
        )
        cab_ok = res.returncode == 0 and os.path.exists(deskthemepack_path)
    if not cab_ok:
        shutil.copy2(zip_bundle_path, deskthemepack_path)

    mockup_img = render_windows11_desktop_mockup(
        primary_wall,
        theme_display_name=clean_title,
        accent_hex=accent_hex,
        palette_hexes=palette_hexes,
        dark_mode=system_dark_mode,
    )
    mockup_path = os.path.join(build_dir, f"{slug}_desktop_preview.png")
    mockup_img.save(mockup_path, format="PNG")

    logon_wav_path = os.path.join(media_dir, event_wav_map["WindowsLogon"])
    elapsed = time.time() - t0
    status_md = (
        f"### ✅ Built Windows 10/11 ThemePack `{clean_title}` in `{elapsed:.1f}s`\n"
        f"- **Dominant Wallpaper Accent**: `{accent_hex}` (`ColorizationColor={accent_dword}` | Mode: `{mode_str}`)\n"
        f"- **Sound Events Packaged**: `21 / 21` `.wav` events (`{matched_from_pack}` from pack, `{21 - matched_from_pack}` synthesized via `{acoustic_palette}`)\n"
        f"- **`.deskthemepack` File**: `{deskthemepack_path}`\n"
        f"- **Complete Installer `.zip` (with `Install_Complete_Theme.ps1` & `.reg`)**: `{zip_bundle_path}`"
    )
    return mockup_img, logon_wav_path, deskthemepack_path, zip_bundle_path, status_md


# ---------------------------------------------------------------------------
# 5. Core Engine 2: OBS Browser-Source HTML5 Soundboard & Stream Deck Exporter
# ---------------------------------------------------------------------------
HOTKEY_DEFAULT_LIST = [
    "F13", "F14", "F15", "F16", "F17", "F18",
    "F19", "F20", "F21", "F22", "F23", "F24",
    "Numpad1", "Numpad2", "Numpad3", "Numpad4", "Numpad5", "Numpad6",
    "Numpad7", "Numpad8", "Numpad9", "Numpad0", "KeyQ", "KeyW",
]

PAD_COLORS = [
    "#3898FF", "#9B58FF", "#00E5B0", "#FF5C8D",
    "#FFB340", "#45D4FF", "#7C6BFF", "#2CEAA3",
]


def render_streamdeck_icon(label: str, hotkey: str, hex_color: str, size: int = 144) -> Image.Image:
    '''Renders a crisp 144×144 or 72×72 neon glass Stream Deck key icon.'''
    hc = hex_color.lstrip("#")
    r, g, b = int(hc[0:2], 16), int(hc[2:4], 16), int(hc[4:6], 16)
    img = Image.new("RGBA", (size, size), (16, 18, 26, 255))
    draw = ImageDraw.Draw(img, "RGBA")

    pad = max(4, size // 18)
    draw.rounded_rectangle(
        [pad, pad, size - pad, size - pad],
        radius=size // 6,
        fill=(24, 28, 40, 255),
        outline=(r, g, b, 235),
        width=max(2, size // 36),
    )
    # Top hotkey badge
    draw.rounded_rectangle(
        [pad + 8, pad + 8, size - pad - 8, pad + int(size * 0.26)],
        radius=6,
        fill=(r, g, b, 55),
    )
    draw.text((pad + 14, pad + 11), hotkey[:10], fill=(r, g, b, 255))

    # Center waveform bars graphic
    cy = int(size * 0.54)
    bars = [0.25, 0.55, 0.9, 0.45, 0.75, 0.35, 0.60]
    bw = max(3, size // 22)
    gap = max(3, size // 28)
    total_w = len(bars) * bw + (len(bars) - 1) * gap
    x_start = (size - total_w) // 2
    for idx, bh_norm in enumerate(bars):
        bx = x_start + idx * (bw + gap)
        bh = int(size * 0.22 * bh_norm)
        draw.rounded_rectangle([bx, cy - bh // 2, bx + bw, cy + bh // 2], radius=2, fill=(r, g, b, 220))

    # Bottom label
    short_lbl = label[:14]
    draw.text((pad + 10, int(size * 0.76)), short_lbl, fill=(235, 240, 255, 255))
    return img.convert("RGB")


def build_obs_soundboard_and_streamdeck(
    board_title: str = "AEI Live Production & Windows SFX Deck",
    drive_sound_zip: str = "(None — Use Procedural 12-Pad SFX Deck)",
    uploaded_wavs: Optional[List[Any]] = None,
    acoustic_palette: str = "Fluent Glass & Crystal Chimes (Windows 11 Modern)",
    grid_columns: int = 4,
    max_pads: int = 12,
) -> Tuple[str, str, Image.Image, str]:
    '''
    Generates:
      1. Self-contained obs_soundboard.html with base64-embedded WAVs, WebAudio FFT visualizer,
         keyboard hotkeys (F13-F24 / Numpad), per-pad volume & loop toggles.
      2. Stream Deck asset pack (.zip) with 72×72 & 144×144 PNG key icons + WAV stems + manifest.
    Returns (html_file_path, streamdeck_zip_path, contact_sheet_pil, status_md)
    '''
    t0 = time.time()
    out_root = globals().get("OUTPUT_DIR", "/content/drive/MyDrive/AEI_TTS_Out/Windows_ThemePacks")
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    slug = re.sub(r"[^A-Za-z0-9_-]+", "_", (board_title or "Soundboard").strip()).strip("_") or "Soundboard"
    deck_dir = os.path.join(out_root, f"soundboard_{slug}_{ts}")
    icons_72_dir = os.path.join(deck_dir, "StreamDeck_Icons_72x72")
    icons_144_dir = os.path.join(deck_dir, "StreamDeck_Icons_144x144")
    audio_dir = os.path.join(deck_dir, "Audio")
    for d in [deck_dir, icons_72_dir, icons_144_dir, audio_dir]:
        os.makedirs(d, exist_ok=True)

    pad_entries: List[Tuple[str, str]] = []
    temp_dir = tempfile.mkdtemp(prefix="aei_sb_")

    if uploaded_wavs:
        for uf in uploaded_wavs[:max_pads]:
            p = uf.name if hasattr(uf, "name") else str(uf)
            if os.path.exists(p) and p.lower().endswith((".wav", ".mp3", ".ogg", ".flac")):
                pad_entries.append((os.path.splitext(os.path.basename(p))[0], p))

    if not pad_entries and drive_sound_zip and os.path.exists(drive_sound_zip) and zipfile.is_zipfile(drive_sound_zip):
        with zipfile.ZipFile(drive_sound_zip, "r") as zf:
            zf.extractall(temp_dir)
        for r_dir, _, files in os.walk(temp_dir):
            for fn in sorted(files):
                if fn.lower().endswith(".wav"):
                    pad_entries.append((os.path.splitext(fn)[0], os.path.join(r_dir, fn)))
                    if len(pad_entries) >= max_pads:
                        break

    if not pad_entries:
        for ev in WINDOWS_21_EVENTS[:max_pads]:
            wav_arr = synthesize_event_wav(ev, palette_name=acoustic_palette, sr=44100)
            p = os.path.join(temp_dir, f"{ev['id'].lstrip('.')}.wav")
            sf.write(p, wav_arr, 44100, subtype="PCM_16")
            pad_entries.append((ev["label"], p))

    pad_entries = pad_entries[:max_pads]
    pads_json_data = []
    icon_144_list: List[Image.Image] = []

    for idx, (lbl, src_wav) in enumerate(pad_entries):
        hotkey = HOTKEY_DEFAULT_LIST[idx % len(HOTKEY_DEFAULT_LIST)]
        color_hex = PAD_COLORS[idx % len(PAD_COLORS)]
        safe_lbl = re.sub(r"[^A-Za-z0-9_-]+", "_", lbl).strip("_") or f"pad_{idx+1:02d}"
        dst_wav = os.path.join(audio_dir, f"{idx+1:02d}_{safe_lbl}.wav")
        shutil.copy2(src_wav, dst_wav)

        with open(dst_wav, "rb") as bf:
            b64_audio = base64.b64encode(bf.read()).decode("ascii")

        ic144 = render_streamdeck_icon(lbl, hotkey, color_hex, size=144)
        ic72 = ic144.resize((72, 72), Image.Resampling.LANCZOS)
        ic144.save(os.path.join(icons_144_dir, f"{idx+1:02d}_{safe_lbl}@2x.png"))
        ic72.save(os.path.join(icons_72_dir, f"{idx+1:02d}_{safe_lbl}.png"))
        icon_144_list.append(ic144)

        pads_json_data.append({
            "id": idx + 1,
            "label": lbl[:22],
            "hotkey": hotkey,
            "color": color_hex,
            "dataUri": f"data:audio/wav;base64,{b64_audio}",
        })

    shutil.rmtree(temp_dir, ignore_errors=True)

    # Build Contact Sheet of Stream Deck Icons
    cols = max(1, min(6, int(grid_columns)))
    rows = math.ceil(len(icon_144_list) / cols)
    sheet_w = cols * 160 + 20
    sheet_h = rows * 160 + 20
    contact_sheet = Image.new("RGB", (sheet_w, sheet_h), (12, 14, 20))
    for idx, ic in enumerate(icon_144_list):
        r_i = idx // cols
        c_i = idx % cols
        contact_sheet.paste(ic, (16 + c_i * 160, 16 + r_i * 160))

    # Build Zero-Dependency Self-Contained HTML5 OBS Soundboard
    pads_js_str = json.dumps(pads_json_data)
    html_doc = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>{board_title}</title>
<style>
  :root {{
    --bg: #0c0e16;
    --card: rgba(24, 28, 42, 0.88);
    --text: #eef2ff;
    --sub: #94a3b8;
  }}
  * {{ box-sizing: border-box; user-select: none; }}
  body {{
    margin: 0; padding: 20px;
    background: radial-gradient(circle at 50% 0%, #1a2238 0%, #0b0d14 75%);
    color: var(--text);
    font-family: 'Segoe UI', system-ui, -apple-system, sans-serif;
    min-height: 100vh;
  }}
  header {{
    display: flex; flex-wrap: wrap; align-items: center; justify-content: space-between;
    gap: 14px; margin-bottom: 18px; padding: 14px 20px;
    background: var(--card); border-radius: 14px;
    border: 1px solid rgba(255,255,255,0.08);
  }}
  h1 {{ font-size: 1.25rem; margin: 0; letter-spacing: 0.4px; }}
  .controls {{ display: flex; align-items: center; gap: 14px; }}
  .panic-btn {{
    background: #ef4444; color: #fff; border: none; border-radius: 8px;
    padding: 8px 16px; font-weight: 600; cursor: pointer;
  }}
  canvas#viz {{
    width: 240px; height: 42px; background: rgba(0,0,0,0.35); border-radius: 8px;
  }}
  .grid {{
    display: grid;
    grid-template-columns: repeat({cols}, minmax(150px, 1fr));
    gap: 14px;
  }}
  .pad {{
    position: relative;
    background: var(--card);
    border: 2px solid rgba(255,255,255,0.12);
    border-radius: 14px;
    padding: 16px;
    cursor: pointer;
    transition: transform 0.08s ease, box-shadow 0.12s ease;
    display: flex; flex-direction: column; justify-content: space-between;
    min-height: 125px;
  }}
  .pad:active, .pad.playing {{
    transform: scale(0.97);
    box-shadow: 0 0 22px var(--pad-color, #3898FF);
    border-color: var(--pad-color, #3898FF);
  }}
  .pad-top {{ display: flex; justify-content: space-between; align-items: center; font-size: 0.78rem; color: var(--sub); }}
  .hotkey {{
    background: rgba(255,255,255,0.09); padding: 3px 8px; border-radius: 6px;
    font-family: monospace; color: var(--pad-color, #3898FF); font-weight: 700;
  }}
  .pad-label {{ font-size: 1.0rem; font-weight: 600; margin: 12px 0 8px 0; }}
  .pad-bottom {{ display: flex; align-items: center; justify-content: space-between; gap: 8px; }}
  .loop-toggle {{
    font-size: 0.72rem; padding: 3px 7px; border-radius: 5px;
    background: rgba(255,255,255,0.07); color: var(--sub); border: none; cursor: pointer;
  }}
  .loop-toggle.active {{ background: var(--pad-color, #3898FF); color: #000; font-weight: 700; }}
</style>
</head>
<body>
  <header>
    <div>
      <h1>🎛️ {board_title}</h1>
      <small style="color:var(--sub)">Click any pad or press its hotkey (F13–F24 / Numpad) • Space / Esc stops all</small>
    </div>
    <div class="controls">
      <canvas id="viz" width="240" height="42"></canvas>
      <label style="font-size:0.85rem">Master <input id="masterVol" type="range" min="0" max="1" step="0.02" value="0.9" /></label>
      <button class="panic-btn" onclick="stopAll()">⏹ STOP ALL (Esc)</button>
    </div>
  </header>
  <div class="grid" id="padGrid"></div>

<script>
const PADS = {pads_js_str};
let audioCtx = null;
let masterGain = null;
let analyser = null;
const activeSources = new Map();
const loopState = new Map();

function ensureAudio() {{
  if (!audioCtx) {{
    audioCtx = new (window.AudioContext || window.webkitAudioContext)();
    masterGain = audioCtx.createGain();
    analyser = audioCtx.createAnalyser();
    analyser.fftSize = 64;
    masterGain.connect(analyser);
    analyser.connect(audioCtx.destination);
    masterGain.gain.value = parseFloat(document.getElementById('masterVol').value);
    requestAnimationFrame(drawViz);
  }}
  if (audioCtx.state === 'suspended') audioCtx.resume();
}}

document.getElementById('masterVol').addEventListener('input', (e) => {{
  if (masterGain) masterGain.gain.value = parseFloat(e.target.value);
}});

async function triggerPad(pad) {{
  ensureAudio();
  const el = document.getElementById('pad-' + pad.id);
  if (activeSources.has(pad.id)) {{
    try {{ activeSources.get(pad.id).stop(); }} catch(e) {{}}
    activeSources.delete(pad.id);
  }}
  const resp = await fetch(pad.dataUri);
  const buf = await resp.arrayBuffer();
  const audioBuf = await audioCtx.decodeAudioData(buf);
  const src = audioCtx.createBufferSource();
  src.buffer = audioBuf;
  src.loop = !!loopState.get(pad.id);
  src.connect(masterGain);
  el.classList.add('playing');
  src.onended = () => {{
    el.classList.remove('playing');
    activeSources.delete(pad.id);
  }};
  activeSources.set(pad.id, src);
  src.start(0);
}}

function stopAll() {{
  for (const [id, src] of activeSources.entries()) {{
    try {{ src.stop(); }} catch(e) {{}}
    const el = document.getElementById('pad-' + id);
    if (el) el.classList.remove('playing');
  }}
  activeSources.clear();
}}

function toggleLoop(ev, id) {{
  ev.stopPropagation();
  const next = !loopState.get(id);
  loopState.set(id, next);
  ev.target.classList.toggle('active', next);
}}

function renderGrid() {{
  const grid = document.getElementById('padGrid');
  PADS.forEach(p => {{
    const div = document.createElement('div');
    div.className = 'pad';
    div.id = 'pad-' + p.id;
    div.style.setProperty('--pad-color', p.color);
    div.onclick = () => triggerPad(p);
    div.innerHTML = `
      <div class="pad-top">
        <span>PAD #${{String(p.id).padStart(2, '0')}}</span>
        <span class="hotkey">${{p.hotkey}}</span>
      </div>
      <div class="pad-label">${{p.label}}</div>
      <div class="pad-bottom">
        <button class="loop-toggle" onclick="toggleLoop(event, ${{p.id}})">🔁 LOOP</button>
      </div>
    `;
    grid.appendChild(div);
  }});
}}

window.addEventListener('keydown', (e) => {{
  if (e.code === 'Escape' || e.code === 'Space') {{
    e.preventDefault();
    stopAll();
    return;
  }}
  const match = PADS.find(p => p.hotkey.toLowerCase() === e.code.toLowerCase() || p.hotkey.toLowerCase() === e.key.toLowerCase());
  if (match) {{
    e.preventDefault();
    triggerPad(match);
  }}
}});

function drawViz() {{
  requestAnimationFrame(drawViz);
  if (!analyser) return;
  const cvs = document.getElementById('viz');
  const ctx = cvs.getContext('2d');
  const data = new Uint8Array(analyser.frequencyBinCount);
  analyser.getByteFrequencyData(data);
  ctx.clearRect(0, 0, cvs.width, cvs.height);
  const bw = (cvs.width / data.length) - 2;
  for (let i = 0; i < data.length; i++) {{
    const h = (data[i] / 255) * cvs.height;
    ctx.fillStyle = '#3898FF';
    ctx.fillRect(i * (bw + 2), cvs.height - h, bw, h);
  }}
}}

renderGrid();
</script>
</body>
</html>
'''
    html_path = os.path.join(deck_dir, "obs_soundboard.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html_doc)

    manifest_path = os.path.join(deck_dir, "StreamDeck_Manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "board_title": board_title,
                "created_at": ts,
                "pad_count": len(pad_entries),
                "pads": [
                    {
                        "id": p["id"],
                        "label": p["label"],
                        "hotkey": p["hotkey"],
                        "icon_72": f"StreamDeck_Icons_72x72/{p['id']:02d}_{re.sub(r'[^A-Za-z0-9_-]+', '_', p['label']).strip('_')}.png",
                        "icon_144": f"StreamDeck_Icons_144x144/{p['id']:02d}_{re.sub(r'[^A-Za-z0-9_-]+', '_', p['label']).strip('_')}@2x.png",
                    }
                    for p in pads_json_data
                ],
            },
            f,
            indent=2,
        )

    zip_path = os.path.join(out_root, f"StreamDeck_OBS_Soundboard_{slug}_{ts}.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for r_dir, _, files in os.walk(deck_dir):
            for fn in files:
                full_p = os.path.join(r_dir, fn)
                zf.write(full_p, arcname=os.path.relpath(full_p, deck_dir))

    elapsed = time.time() - t0
    status_md = (
        f"### ✅ Generated OBS HTML5 Soundboard & Stream Deck Pack in `{elapsed:.1f}s`\n"
        f"- **Interactive Pads**: `{len(pad_entries)}` pads with embedded base64 WebAudio & hotkeys (`F13–F24` / `Numpad`)\n"
        f"- **Standalone OBS Browser Source File**: `{html_path}`\n"
        f"- **Stream Deck Bundle (`72×72` + `144×144` PNG Icons + WAVs + HTML)**: `{zip_path}`"
    )
    return html_path, zip_path, contact_sheet, status_md


# ---------------------------------------------------------------------------
# 6. Core Engine 3: 1080p Animated Desktop & Sound Scheme Video Showcase
# ---------------------------------------------------------------------------
def render_theme_showcase_video(
    theme_name: str = "AEI Fluent Cyber Glass",
    acoustic_palette: str = "Fluent Glass & Crystal Chimes (Windows 11 Modern)",
    wallpaper_style: str = "Fluent Sapphire & Violet Acrylic Waves (2560×1440)",
    num_showcase_events: int = 6,
) -> Tuple[str, str]:
    '''Renders a 1080p MP4 video previewing the Windows 11 desktop mockup & toast popups synchronized with the sound events.'''
    t0 = time.time()
    out_root = globals().get("OUTPUT_DIR", "/content/drive/MyDrive/AEI_TTS_Out/Windows_ThemePacks")
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    slug = re.sub(r"[^A-Za-z0-9_-]+", "_", (theme_name or "Theme").strip()).strip("_") or "Theme"

    wall = generate_procedural_wallpaper(style_name=wallpaper_style, width=1920, height=1080)
    accent_hex, _, palette_hexes = extract_windows_accent_color(wall)

    events_subset = WINDOWS_21_EVENTS[: max(2, min(10, int(num_showcase_events)))]
    sr = 44100
    fps = 15
    audio_segments = []
    temp_vdir = tempfile.mkdtemp(prefix="aei_vid_")
    frame_idx = 0

    for ev in events_subset:
        wav_arr = synthesize_event_wav(ev, palette_name=acoustic_palette, sr=sr)
        pad_silence = np.zeros((int(0.35 * sr), 2), dtype=np.float32)
        seg = np.concatenate([wav_arr, pad_silence], axis=0)
        audio_segments.append(seg)

        seg_dur = seg.shape[0] / sr
        n_frames = max(1, int(round(seg_dur * fps)))
        mockup = render_windows11_desktop_mockup(
            wall,
            theme_display_name=theme_name,
            accent_hex=accent_hex,
            palette_hexes=palette_hexes,
            dark_mode=True,
            active_event_label=f"{ev['id']} — {ev['label']}",
        )
        for _ in range(n_frames):
            frame_p = os.path.join(temp_vdir, f"frame_{frame_idx:05d}.png")
            mockup.save(frame_p, format="PNG")
            frame_idx += 1

    full_audio = np.concatenate(audio_segments, axis=0)
    audio_path = os.path.join(temp_vdir, "showcase_audio.wav")
    sf.write(audio_path, full_audio, sr, subtype="PCM_16")

    mp4_path = os.path.join(out_root, f"{slug}_Showcase_{ts}.mp4")
    cmd = [
        "ffmpeg", "-y",
        "-framerate", str(fps),
        "-i", os.path.join(temp_vdir, "frame_%05d.png"),
        "-i", audio_path,
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "ultrafast",
        "-c:a", "aac", "-b:a", "192k",
        "-shortest",
        mp4_path,
    ]
    subprocess.run(cmd, check=False, capture_output=True)
    shutil.rmtree(temp_vdir, ignore_errors=True)

    elapsed = time.time() - t0
    md = (
        f"### ✅ Rendered Desktop Theme Showcase Video (`{elapsed:.1f}s`)\n"
        f"- **Events Featured**: `{len(events_subset)}` sequential Windows sound events\n"
        f"- **Saved MP4**: `{mp4_path}`"
    )
    return mp4_path, md


print("✅ Windows 10/11 ThemePack & Soundboard Core Engines Initialized!")


In [ ]:
# @title STEP 4: Launch Gradio 5.49.1 Windows ThemePack & OBS Soundboard Studio { display-mode: "form" }
# @markdown Launches the 4-tab UI for building `.deskthemepack` desktop themes, OBS HTML5 soundboards, Elgato Stream Deck key icons, and animated theme showcase videos.

import os
import gradio as gr
from IPython.display import clear_output

def _ui_refresh_all():
    packs = ["(None — Synthesize Procedural 21-Event Scheme)"] + discover_drive_sound_packs()
    walls = ["(None — Generate Procedural 2K Wallpaper)"] + discover_drive_wallpapers()
    return gr.update(choices=packs), gr.update(choices=walls), gr.update(choices=packs)

init_packs = ["(None — Synthesize Procedural 21-Event Scheme)"] + discover_drive_sound_packs()
init_walls = ["(None — Generate Procedural 2K Wallpaper)"] + discover_drive_wallpapers()

ACOUSTIC_PALETTES = [
    "Fluent Glass & Crystal Chimes (Windows 11 Modern)",
    "Cyberpunk Neon FM Synth & Sub Glitch",
    "Warm Acoustic Marimba & Felt Piano",
    "Sci-Fi Starship Bridge Telemetry",
    "Minimalist Tactile Micro-Clicks",
]

WALLPAPER_STYLES = [
    "Fluent Sapphire & Violet Acrylic Waves (2560×1440)",
    "Cyberpunk Midnight Neon Horizon (2560×1440)",
    "Nordic Aurora Borealis Dark Slate (2560×1440)",
    "Solar Gold & Obsidian Geometric Glass (2560×1440)",
]

with gr.Blocks(title="Windows 10/11 ThemePack & Soundboard Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        '''
        # 🪟 Windows 10/11 `.deskthemepack` & Stream Deck / OBS Interactive Soundboard Studio
        **Complete 21-Event Windows Sound Schemes · Dominant Wallpaper Accent Color Extractor · Self-Contained `obs_soundboard.html` · Stream Deck Key Icons**
        '''
    )
    welcome_md = gr.Markdown()

    with gr.Tabs():
        # ===================================================================
        # TAB 1: Windows 10/11 .theme & .deskthemepack Builder
        # ===================================================================
        with gr.TabItem("🪟 1. Windows 10/11 .deskthemepack & .theme Builder"):
            with gr.Row():
                with gr.Column(scale=5):
                    t1_name = gr.Textbox(
                        label="Theme Display Name",
                        value="AEI Fluent Cyber Glass",
                        info="Appears directly in Windows Settings -> Personalization -> Themes.",
                    )
                    with gr.Row():
                        t1_sound_zip = gr.Dropdown(
                            label="Select Windows Sound Scheme (.zip) from Google Drive",
                            choices=init_packs,
                            value=init_packs[0],
                            info="Auto-discovers sound packs from Stable-Audio-3, MOSS-SoundEffect-v2, AudioX-Turbo, MiDashengLM-Gen, HunyuanVideo-Foley & TangoFlux.",
                        )
                        t1_refresh = gr.Button("🔄 Refresh Drive")
                    t1_sound_uploads = gr.File(
                        label="Or Upload Custom .wav Stems / Sound Scheme .zip",
                        file_count="multiple",
                        file_types=[".wav", ".zip"],
                    )
                    t1_palette = gr.Dropdown(
                        label="Procedural Acoustic Palette (Used for Any Unmapped Events)",
                        choices=ACOUSTIC_PALETTES,
                        value=ACOUSTIC_PALETTES[0],
                        info="Guarantees all 21 Windows sound events are populated even if you only upload a few custom WAVs.",
                    )
                    with gr.Accordion("🖼️ Desktop Wallpaper & Dominant Accent Color Settings", open=True):
                        t1_wall_upload = gr.Image(
                            label="Upload Primary Desktop Wallpaper (Optional)",
                            type="pil",
                            info="Upload a 16:9 wallpaper or pick one generated by Qwen-Image-2.1 below.",
                        )
                        t1_wall_drive = gr.Dropdown(
                            label="Or Pick Wallpaper Image from Google Drive (AEI_3D_Out)",
                            choices=init_walls,
                            value=init_walls[0],
                            info="Scans AEI_3D_Out for 2K wallpapers generated by Qwen-Image-2.1.",
                        )
                        t1_wall_style = gr.Dropdown(
                            label="Procedural 2560×1440 Wallpaper Style (When No Image Uploaded)",
                            choices=WALLPAPER_STYLES,
                            value=WALLPAPER_STYLES[0],
                            info="Synthesizes a 2K wallpaper if no custom image is selected.",
                        )
                        t1_extra_walls = gr.File(
                            label="Additional Slideshow Wallpapers (Optional)",
                            file_count="multiple",
                            file_types=["image"],
                        )
                        with gr.Row():
                            t1_accent_hex = gr.Textbox(
                                label="Override Windows Accent Hex (Leave Blank for Auto-Extraction)",
                                placeholder="#3898FF",
                                info="When blank, automatically extracts the dominant vibrant ColorizationColor from your wallpaper.",
                            )
                            t1_dark_mode = gr.Checkbox(
                                label="Windows Dark System & App Mode",
                                value=True,
                                info="Sets SystemMode=Dark and AppMode=Dark in [VisualStyles].",
                            )
                            t1_interval = gr.Slider(
                                label="Slideshow Interval (Minutes)",
                                minimum=1,
                                maximum=120,
                                step=1,
                                value=30,
                                info="Wallpaper rotation interval in minutes.",
                            )
                    t1_btn = gr.Button("🪟 Build .deskthemepack + .theme + 1-Click PowerShell Installer", variant="primary")

                with gr.Column(scale=5):
                    t1_mockup = gr.Image(label="Simulated Windows 11 Desktop, Taskbar & Extracted Palette Preview", type="pil")
                    t1_audio_prev = gr.Audio(label="Preview WindowsLogon Startup Sound (.wav)", type="filepath")
                    with gr.Row():
                        t1_out_desk = gr.File(label="Download .deskthemepack")
                        t1_out_zip = gr.File(label="Download Complete Installer Bundle (.zip)")
                    t1_status = gr.Markdown()

            t1_btn.click(
                fn=build_windows_themepack_bundle,
                inputs=[
                    t1_name, t1_sound_zip, t1_sound_uploads, t1_palette,
                    t1_wall_upload, t1_wall_drive, t1_wall_style, t1_extra_walls,
                    t1_accent_hex, t1_dark_mode, t1_interval,
                ],
                outputs=[t1_mockup, t1_audio_prev, t1_out_desk, t1_out_zip, t1_status],
                concurrency_limit=1,
            )

        # ===================================================================
        # TAB 2: OBS Browser-Source Soundboard & Elgato Stream Deck Exporter
        # ===================================================================
        with gr.TabItem("🎛️ 2. OBS HTML5 Soundboard & Elgato Stream Deck Packager"):
            with gr.Row():
                with gr.Column(scale=5):
                    t2_title = gr.Textbox(
                        label="Soundboard Title",
                        value="AEI Live Production & Windows SFX Deck",
                        info="Header title displayed inside obs_soundboard.html.",
                    )
                    t2_zip = gr.Dropdown(
                        label="Source Sound Pack (.zip) from Google Drive",
                        choices=init_packs,
                        value=init_packs[0],
                        info="Select any generated sound pack or use procedural pads.",
                    )
                    t2_wavs = gr.File(
                        label="Or Upload Individual Audio Stems (.wav / .mp3 / .ogg)",
                        file_count="multiple",
                        file_types=[".wav", ".mp3", ".ogg", ".flac"],
                    )
                    t2_palette = gr.Dropdown(
                        label="Fallback Procedural Acoustic Palette",
                        choices=ACOUSTIC_PALETTES,
                        value=ACOUSTIC_PALETTES[0],
                        info="Used when generating a standalone demo soundboard.",
                    )
                    with gr.Row():
                        t2_cols = gr.Slider(
                            label="Soundboard Grid Columns",
                            minimum=2,
                            maximum=6,
                            step=1,
                            value=4,
                            info="Number of columns in obs_soundboard.html.",
                        )
                        t2_max_pads = gr.Slider(
                            label="Maximum Sound Pads",
                            minimum=4,
                            maximum=24,
                            step=1,
                            value=12,
                            info="Up to 24 interactive pads mapped to F13–F24 and Numpad keys.",
                        )
                    t2_btn = gr.Button("🎛️ Generate obs_soundboard.html & Stream Deck Icon Pack (.zip)", variant="primary")

                with gr.Column(scale=5):
                    t2_sheet = gr.Image(label="Generated Elgato Stream Deck Key Icons (144×144 @2x Preview)", type="pil")
                    with gr.Row():
                        t2_html_file = gr.File(label="Download Standalone obs_soundboard.html")
                        t2_zip_file = gr.File(label="Download Complete Stream Deck + OBS Bundle (.zip)")
                    t2_status = gr.Markdown()

            t2_btn.click(
                fn=build_obs_soundboard_and_streamdeck,
                inputs=[t2_title, t2_zip, t2_wavs, t2_palette, t2_cols, t2_max_pads],
                outputs=[t2_html_file, t2_zip_file, t2_sheet, t2_status],
                concurrency_limit=1,
            )

        # ===================================================================
        # TAB 3: Animated Desktop & Sound Scheme Showcase Video Renderer
        # ===================================================================
        with gr.TabItem("🎬 3. 1080p Desktop Theme & Sound Scheme Video Showcase"):
            with gr.Row():
                with gr.Column(scale=5):
                    t3_name = gr.Textbox(
                        label="Showcase Theme Title",
                        value="AEI Fluent Cyber Glass",
                        info="Displayed on the simulated Windows 11 notification toast.",
                    )
                    t3_palette = gr.Dropdown(
                        label="Acoustic Palette",
                        choices=ACOUSTIC_PALETTES,
                        value=ACOUSTIC_PALETTES[0],
                        info="Sound palette played sequentially in the showcase video.",
                    )
                    t3_wall = gr.Dropdown(
                        label="Desktop Wallpaper Style",
                        choices=WALLPAPER_STYLES,
                        value=WALLPAPER_STYLES[0],
                        info="Background wallpaper rendered in the 1080p showcase.",
                    )
                    t3_count = gr.Slider(
                        label="Number of Sequential Sound Events to Showcase",
                        minimum=3,
                        maximum=10,
                        step=1,
                        value=6,
                        info="Each event triggers a toast update and plays its stereo chime.",
                    )
                    t3_btn = gr.Button("🎬 Render 1080p Theme Showcase Video (.mp4)", variant="primary")

                with gr.Column(scale=5):
                    t3_video = gr.Video(label="Rendered 1080p Desktop Theme & Sound Showcase (.mp4)")
                    t3_status = gr.Markdown()

            t3_btn.click(
                fn=render_theme_showcase_video,
                inputs=[t3_name, t3_palette, t3_wall, t3_count],
                outputs=[t3_video, t3_status],
                concurrency_limit=1,
            )

    t1_refresh.click(fn=_ui_refresh_all, outputs=[t1_sound_zip, t1_wall_drive, t2_zip])

    def _on_load():
        packs = discover_drive_sound_packs()
        walls = discover_drive_wallpapers()
        return (
            f"✅ **ThemePack & Soundboard Studio Ready** | Discovered `{len(packs)}` Sound Pack `.zip` archives and "
            f"`{len(walls)}` Wallpaper images across `/content/drive/MyDrive/AEI_TTS_Out` & `AEI_3D_Out`."
        )

    demo.load(fn=_on_load, outputs=[welcome_md])

clear_output(wait=True)
demo.queue(default_concurrency_limit=1).launch(share=True, inline=True, show_error=True)


In [ ]:
# @title STEP 5: Colab Session Keep-Alive Utility (Optional) { display-mode: "form" }
# @markdown Optional heartbeat timer to keep the Gradio soundboard server alive during streaming or batch exports.

import time

KEEP_ALIVE_MINUTES = 0  # @param {type:"slider", min:0, max:120, step:5}

if KEEP_ALIVE_MINUTES > 0:
    print(f"⏱️ Keep-alive active for {KEEP_ALIVE_MINUTES} minutes...")
    for m in range(KEEP_ALIVE_MINUTES):
        time.sleep(60)
        print(f"  [Heartbeat] {m + 1}/{KEEP_ALIVE_MINUTES} min elapsed.")
else:
    print("ℹ️ Keep-alive timer set to 0 minutes (idle).")


In [ ]:
# @title STEP 6: End-to-End Quick Test (.deskthemepack + OBS HTML5 Soundboard Smoke Test) { display-mode: "form" }
# @markdown Builds a complete 21-event Windows 10/11 `.deskthemepack` and a 12-pad `obs_soundboard.html` bundle, displays the simulated Windows 11 desktop preview inline, and provides downloadable `FileLink`s.

from IPython.display import display, FileLink

QUICK_THEME_NAME = "AEI Sapphire Glass Edition"  # @param {type:"string"}

mockup_img, logon_wav, desk_path, zip_path, status_txt = build_windows_themepack_bundle(
    theme_name=QUICK_THEME_NAME,
    acoustic_palette="Fluent Glass & Crystal Chimes (Windows 11 Modern)",
    procedural_wallpaper_style="Fluent Sapphire & Violet Acrylic Waves (2560×1440)",
)
print(status_txt)
display(mockup_img.resize((640, 360)))
display(FileLink(desk_path))
display(FileLink(zip_path))

html_p, sb_zip, sheet_img, sb_status = build_obs_soundboard_and_streamdeck(
    board_title=QUICK_THEME_NAME + " Soundboard",
    max_pads=8,
)
print(sb_status)
display(FileLink(html_p))
display(FileLink(sb_zip))


In [ ]:
# @title STEP 7: Batch Convert All Drive Sound Packs into Complete .deskthemepack & OBS Soundboard Bundles { display-mode: "form" }
# @markdown Iterates over existing `.zip` sound packs in `/content/drive/MyDrive/AEI_TTS_Out` (or synthesizes all 4 acoustic palettes) and packages each into a `.deskthemepack` + `obs_soundboard.html` archive.

import os
from tqdm.auto import tqdm
from IPython.display import display, FileLink

BATCH_SOURCE_MODE = "All 4 Acoustic & Wallpaper Palettes (Standalone Batch)"  # @param ["All 4 Acoustic & Wallpaper Palettes (Standalone Batch)", "All Discovered Sound Pack .zip Archives in Google Drive"]

batch_outputs = []

if "Discovered" in BATCH_SOURCE_MODE:
    zips = discover_drive_sound_packs()[:10]
    if not zips:
        print("ℹ️ No existing .zip sound packs found in Drive; falling back to 4 Acoustic Palettes.")
        BATCH_SOURCE_MODE = "All 4 Acoustic & Wallpaper Palettes (Standalone Batch)"
    else:
        for idx, zp in enumerate(tqdm(zips, desc="Batch ThemePacks")):
            try:
                base_name = os.path.splitext(os.path.basename(zp))[0][:28]
                _, _, desk_p, full_zip, _ = build_windows_themepack_bundle(
                    theme_name=f"AEI_{base_name}",
                    drive_sound_zip=zp,
                    procedural_wallpaper_style=WALLPAPER_STYLES[idx % len(WALLPAPER_STYLES)],
                )
                batch_outputs.append(full_zip)
            except Exception as e:
                print(f"⚠️ Error packaging {zp}: {e}")

if "Standalone" in BATCH_SOURCE_MODE:
    pairs = list(zip(ACOUSTIC_PALETTES[:4], WALLPAPER_STYLES[:4]))
    for idx, (pal, w_style) in enumerate(tqdm(pairs, desc="Batch Palettes")):
        try:
            short_title = pal.split("(")[0].strip()
            _, _, desk_p, full_zip, _ = build_windows_themepack_bundle(
                theme_name=f"AEI {short_title}",
                acoustic_palette=pal,
                procedural_wallpaper_style=w_style,
            )
            batch_outputs.append(full_zip)
        except Exception as e:
            print(f"⚠️ Error on palette {pal}: {e}")

print(f"✅ Completed {len(batch_outputs)} ThemePack bundles!")
for bp in batch_outputs:
    display(FileLink(bp))
